# Exp7.2.6 — Output Readout Loss & Representation Shaping

Analysis-only notebook. It reads finalized aggregate artifacts; training, checkpoint loading, inference, gain calibration, and Slurm dispatch are intentionally excluded.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start=None):
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_7_2_6_output_readout_loss_shaping' / 'output_readout_loss_shaping_v1'
ROOT


## Finalized inputs


In [ ]:
manifest = json.loads((ROOT / 'manifest.json').read_text())
cross = pd.read_csv(ROOT / 'crosscheck_2x2_summary.csv')
ladder = pd.read_csv(ROOT / 'mechanism_ladder_summary.csv')
beta = pd.read_csv(ROOT / 'beta_sweep_summary.csv')
controls = pd.read_csv(ROOT / 'cap_polarity_controls.csv')
e2e = pd.read_csv(ROOT / 'e2e_performance_summary.csv')
probes = pd.read_csv(ROOT / 'e2e_l2_probe_summary.csv')
deltas = pd.read_csv(ROOT / 'representation_shaping_delta_summary.csv')
manifest


## Part I — Frozen-L2 decoding mechanism


In [ ]:
display(cross[cross.regularization == 'task_only'])
display(ladder[ladder.regularization == 'task_only'])


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for regularization, group in beta.groupby('regularization'):
    ordered = group.sort_values('beta')
    ax.plot(ordered.beta, ordered.balanced_accuracy_mean, marker='o', label=regularization)
ax.set_xlabel('Output membrane beta')
ax.set_ylabel('Balanced accuracy')
ax.set_ylim(0, 1)
ax.set_title('Leakage curve')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
display(controls)


## Part II — E2E representation shaping


In [ ]:
display(e2e)
display(probes)
display(deltas)


In [ ]:
test_probe = probes[probes.split == 'test'].copy()
comparison = test_probe.pivot_table(
    index=['regularization', 'probe'],
    columns='condition',
    values='balanced_accuracy_mean',
    aggfunc='first',
).reset_index()
display(comparison)
